# League Analysis

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

model_aliases = {
    "fixed_2p": "DQN Fixed",
    "fixed_3p": "DQN Fixed",
    "fixed_4p": "DQN Fixed",
    "variable_2_4": "DQN Variable",
    "variable_encoder_2_4": "DQN Variable + Encoder",
    "random": "Random",
}
model_order = ["DQN Fixed", "DQN Variable", "DQN Variable + Encoder", "Random"]

result_roots = [Path("results/league_results"), Path("../../results/league_results")]
results_root = next(path for path in result_roots if path.exists())

frames = []
for player_count in (2, 3, 4):
    path = results_root / f"{player_count}p" / "matches.csv"
    if not path.exists():
        continue
    data = pd.read_csv(path)
    data["player_count"] = player_count
    frames.append(data)

df = pd.concat(frames, ignore_index=True)


def split_pipe(value):
    return str(value).split("|")


def pipe_ints(value):
    return [int(part) for part in split_pipe(value)]


def mean_pipe_ints(value):
    values = pipe_ints(value)
    return sum(values) / len(values)


def average_rank_from_points(points, opponent_points_all):
    table_points = [int(points), *pipe_ints(opponent_points_all)]
    higher_scores = sum(score > points for score in table_points)
    tied_scores = sum(score == points for score in table_points)
    return higher_scores + (tied_scores + 1) / 2


df["competitor_label"] = df["competitor"].map(model_aliases)
df["opponent_labels"] = df["opponent_competitors"].apply(
    lambda names: [model_aliases[name] for name in split_pipe(names)]
)
df["opponent_mean_points"] = df["opponent_points_all"].apply(mean_pipe_ints)
df["point_margin"] = df["points"] - df["opponent_mean_points"]
df["average_rank"] = df.apply(
    lambda row: average_rank_from_points(row["points"], row["opponent_points_all"]), axis=1
)
df["won"] = df["outcome"].eq("win")
df["player_count_label"] = df["player_count"].astype(str) + "p"


In [ ]:
df

## Results Loaded

In [ ]:
loaded = (
    df.groupby("player_count")
      .agg(rows=("match_id", "size"), games=("match_id", "nunique"), matchups=("matchup_id", "nunique"))
      .reset_index()
)
loaded

## Overall Results

In [ ]:
overall = (
    df.groupby(["player_count", "competitor_label"])
      .agg(
          games=("match_id", "size"),
          wins=("won", "sum"),
          losses=("outcome", lambda x: x.eq("loss").sum()),
          ties=("outcome", lambda x: x.eq("tie").sum()),
          average_rank=("average_rank", "mean"),
          raw_win_rate=("won", "mean"),
          mean_points=("points", "mean"),
          mean_margin=("point_margin", "mean"),
      )
      .reset_index()
)
overall.style.format({
    "average_rank": "{:.2f}",
    "raw_win_rate": "{:.1%}",
    "mean_points": "{:.2f}",
    "mean_margin": "{:+.2f}",
})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

sns.barplot(
    data=overall,
    x="player_count",
    y="average_rank",
    hue="competitor_label",
    hue_order=model_order,
    errorbar=None,
    ax=axes[0],
)
axes[0].set_title("Average rank")
axes[0].set_xlabel("Players")
axes[0].set_ylabel("Rank (lower is better)")
axes[0].set_ylim(df["player_count"].max() + 0.1, 0.9)

sns.barplot(
    data=overall,
    x="player_count",
    y="raw_win_rate",
    hue="competitor_label",
    hue_order=model_order,
    errorbar=None,
    ax=axes[1],
)
axes[1].set_title("Win rate")
axes[1].set_xlabel("Players")
axes[1].set_ylabel("Win rate")
axes[1].set_ylim(0, 1)
axes[1].yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")

## Matchup View

In [ ]:
opponent_rows = []
for row in df.itertuples(index=False):
    for opponent_label, opponent_points in zip(row.opponent_labels, pipe_ints(row.opponent_points_all)):
        opponent_rows.append({
            "player_count": row.player_count,
            "competitor_label": row.competitor_label,
            "opponent_label": opponent_label,
            "head_to_head_win": row.points > opponent_points,
            "head_to_head_tie": row.points == opponent_points,
            "head_to_head_margin": row.points - opponent_points,
        })

opponent_df = pd.DataFrame(opponent_rows)
matchup_summary = (
    opponent_df.groupby(["player_count", "competitor_label", "opponent_label"])
      .agg(
          rows=("head_to_head_win", "size"),
          head_to_head_wins=("head_to_head_win", "sum"),
          head_to_head_ties=("head_to_head_tie", "sum"),
          head_to_head_losses=("head_to_head_win", lambda wins: (~wins).sum()),
          mean_head_to_head_margin=("head_to_head_margin", "mean"),
      )
      .reset_index()
)
matchup_summary["head_to_head_losses"] = matchup_summary["head_to_head_losses"] - matchup_summary["head_to_head_ties"]
matchup_summary["non_tie_games"] = matchup_summary["head_to_head_wins"] + matchup_summary["head_to_head_losses"]
matchup_summary["non_tie_head_to_head_win_rate"] = (
    matchup_summary["head_to_head_wins"] / matchup_summary["non_tie_games"]
)
matchup_summary.style.format({
    "non_tie_head_to_head_win_rate": "{:.1%}",
    "mean_head_to_head_margin": "{:+.2f}",
})

In [ ]:
for player_count in (2, 3, 4):
    matrix = matchup_summary[matchup_summary["player_count"] == player_count].pivot(
        index="competitor_label", columns="opponent_label", values="non_tie_head_to_head_win_rate"
    ).reindex(index=model_order, columns=model_order)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(matrix, annot=True, fmt=".0%", vmin=0, vmax=1, cmap="RdYlGn", ax=ax)
    ax.set_title(f"{player_count} player head-to-head")
    ax.set_xlabel("Opponent")
    # 45 degree X label
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    ax.set_ylabel("Competitor")
    plt.show()
plt.tight_layout()

## Repetition Variance

In [ ]:
repetition_results = (
    df.groupby(["player_count", "competitor_label", "repetition"])
      .agg(
          games=("match_id", "size"),
          average_rank=("average_rank", "mean"),
          raw_win_rate=("won", "mean"),
      )
      .reset_index()
)

repetition_variance = (
    repetition_results.groupby(["player_count", "competitor_label"])
      .agg(
          repetitions=("repetition", "nunique"),
          mean_average_rank=("average_rank", "mean"),
          average_rank_variance=("average_rank", "var"),
          mean_win_rate=("raw_win_rate", "mean"),
          win_rate_variance=("raw_win_rate", "var"),
      )
      .reset_index()
)
repetition_variance[["average_rank_variance", "win_rate_variance"]] = repetition_variance[
    ["average_rank_variance", "win_rate_variance"]
].fillna(0)
repetition_variance.style.format({
    "mean_average_rank": "{:.2f}",
    "average_rank_variance": "{:.4f}",
    "mean_win_rate": "{:.1%}",
    "win_rate_variance": "{:.4f}",
})


In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(
    data=repetition_variance,
    x="player_count",
    y="average_rank_variance",
    hue="competitor_label",
    hue_order=model_order,
    errorbar=None,
)
plt.title("Average-rank variance between repetitions")
plt.xlabel("Players")
plt.ylabel("Variance")
plt.legend(title="")
plt.tight_layout()


## Seat Effects

In [ ]:
seat_results = (
    df.groupby(["player_count", "competitor_label", "seat"])
      .agg(
          games=("match_id", "size"),
          average_rank=("average_rank", "mean"),
          raw_win_rate=("won", "mean"),
          mean_margin=("point_margin", "mean"),
      )
      .reset_index()
)
seat_results.style.format({
    "average_rank": "{:.2f}",
    "raw_win_rate": "{:.1%}",
    "mean_margin": "{:+.2f}",
})


In [ ]:
grid = sns.catplot(
    data=seat_results,
    kind="point",
    x="seat",
    y="average_rank",
    hue="competitor_label",
    hue_order=model_order,
    col="player_count",
    errorbar=None,
    height=3.5,
    aspect=1.1,
)
grid.set_axis_labels("Seat", "Average rank")
grid.set_titles("{col_name}p")
for ax in grid.axes.flat:
    ax.set_ylim(df["player_count"].max() + 0.1, 0.9)


## Training Metrics

In [ ]:
metrics_candidates = [Path("results/league"), Path("../../results/league")]
metrics_root = next(path for path in metrics_candidates if path.exists())

checkpoint_aliases = {
    "fixed_2p": "DQN Fixed 2 Players",
    "fixed_3p": "DQN Fixed 3 Players",
    "fixed_4p": "DQN Fixed 4 Players",
    "variable_2_4": "DQN Variable",
    "variable_encoder_2_4": "DQN Variable + Encoder",
}

training_metrics = []
for model, alias in checkpoint_aliases.items():
    for metrics_path in sorted((metrics_root / model).glob("repetition_*/metrics.csv")):
        metrics = pd.read_csv(metrics_path)
        metrics["checkpoint"] = alias
        metrics["repetition"] = int(metrics_path.parent.name.removeprefix("repetition_"))
        training_metrics.append(metrics)

training_df = pd.concat(training_metrics, ignore_index=True)
training_df["loss_smoothed"] = training_df.groupby(["checkpoint", "repetition"])["loss"].transform(
    lambda x: x.rolling(100, min_periods=1).mean()
)
training_df["return_smoothed"] = training_df.groupby(["checkpoint", "repetition"])["mean_episode_return"].transform(
    lambda x: x.rolling(100, min_periods=1).mean()
)

In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=training_df, x="iteration", y="loss_smoothed", hue="checkpoint", errorbar="sd")
plt.title("Training loss")
plt.xlabel("Iteration")
plt.ylabel("Loss (100-iteration mean)")
plt.legend(title="", ncol=2)
plt.tight_layout()

In [ ]:
training_time = training_df.groupby(["checkpoint", "repetition"], as_index=False)["elapsed_seconds"].max()
training_time["hours"] = training_time["elapsed_seconds"] / 3600

plt.figure(figsize=(10, 4))
sns.barplot(data=training_time, x="checkpoint", y="hours", errorbar="sd")
plt.title("Total training time")
plt.xlabel("")
plt.ylabel("Hours")
plt.xticks(rotation=15)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=training_df, x="iteration", y="return_smoothed", hue="checkpoint", errorbar="sd")
plt.title("Mean episode return during training")
plt.xlabel("Iteration")
plt.ylabel("Mean episode return (100-iteration mean)")
plt.legend(title="", ncol=2)
plt.tight_layout()